# Unduh Indeks Iklim DMI (IOD) dan ONI (ENSO)

Notebook ini mengunduh dua indeks iklim bulanan dari NOAA PSL lalu menyimpannya ke `data/external/climate_indices.csv`. Format file itu yang dibaca bagian 5 notebook `03_split`.

| Indeks | Fenomena | Sumber | Keterangan |
|---|---|---|---|
| **DMI** (Dipole Mode Index) | IOD | NOAA PSL, dari HadISST1.1 | Anomali SST Samudra Hindia barat (50–70°BT, 10°LS–10°LU) dikurangi tenggara (90–110°BT, 10°LS–0°). Positif berarti laut dekat Sumatra–Jawa lebih dingin. Rujukan: Saji & Yamagata (2003). |
| **ONI** (Oceanic Niño Index) | ENSO | NOAA PSL, dari ERSST.v5 | Rata-rata bergerak 3 bulan anomali SST Niño 3.4. ONI ≥ +0,5 berarti El Niño, ≤ −0,5 berarti La Niña. |

**Kalau unduhan gagal** (misalnya jaringan kampus atau kantor memblokir), buka kedua URL di bawah lewat browser, simpan dengan *Save As* ke `data/external/raw/` memakai nama file yang sama, lalu jalankan ulang notebook. Notebook otomatis memakai file lokal kalau sudah ada.

In [ ]:
import os
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(os.environ.get('THESIS_ROOT', r'D:/thesis'))
RAW_DIR = ROOT / 'data' / 'external' / 'raw'
OUT_PATH = ROOT / 'data' / 'external' / 'climate_indices.csv'
START, END = '2003-01', '2025-12'

SOURCES = {
    'dmi': 'https://psl.noaa.gov/data/timeseries/month/data/dmi.had.long.data',
    'oni': 'https://psl.noaa.gov/data/correlation/oni.data',
}
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Unduh (atau pakai file lokal)

In [ ]:
def get_file(name, url):
    path = RAW_DIR / url.rsplit('/', 1)[-1]
    if path.is_file() and path.stat().st_size > 0:
        print(f'{name}: pakai file lokal {path}')
        return path
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=60) as r:
            path.write_bytes(r.read())
        print(f'{name}: diunduh -> {path}')
        return path
    except Exception as e:
        print(f'{name}: GAGAL mengunduh ({e}).\n  Buka {url} di browser, simpan ke {path}, lalu jalankan ulang.')
        return None

paths = {k: get_file(k, u) for k, u in SOURCES.items()}

## 2. Baca format standar PSL

Format standar PSL: baris pertama berisi tahun awal dan akhir, lalu satu baris per tahun (tahun + 12 nilai bulanan), lalu baris kode *missing* dan keterangan. Nilai yang sangat negatif (misalnya −9999 atau −99,9) adalah kode *missing* dan diubah menjadi NaN.

In [ ]:
def read_psl_standard(path, name):
    lines = Path(path).read_text(errors='ignore').splitlines()
    y0, y1 = (int(float(t)) for t in lines[0].split()[:2])
    recs = []
    for ln in lines[1:]:
        tok = ln.split()
        if len(tok) != 13:
            continue
        try:
            yr = int(float(tok[0])); vals = [float(t) for t in tok[1:]]
        except ValueError:
            continue
        if not (y0 <= yr <= y1):
            continue
        for m, v in enumerate(vals, start=1):
            recs.append((pd.Timestamp(yr, m, 1), np.nan if v <= -99 else v))
    s = pd.Series(dict(recs), name=name).sort_index()
    print(f'{name}: {s.index.min():%Y-%m} s.d. {s.dropna().index.max():%Y-%m}, {int(s.notna().sum())} bulan berisi')
    return s

series = [read_psl_standard(p, k) for k, p in paths.items() if p is not None]
assert series, 'Belum ada file indeks yang terbaca.'
idx = pd.concat(series, axis=1).loc[f'{START}-01':f'{END}-01']
idx.index.name = 'month'
miss = idx.isna().sum()
print('Bulan kosong dalam rentang studi:\n', miss.to_string())

## 3. Simpan dan cek

In [ ]:
out = idx.reset_index()
out['month'] = out.month.dt.strftime('%Y-%m')
out.to_csv(OUT_PATH, index=False)
print('Disimpan ke', OUT_PATH)

ann = idx.copy(); ann['year'] = ann.index.year
son = ann[ann.index.month.isin([9, 10, 11])].groupby('year').mean().round(2)
son.columns = [f'{c}_SON' for c in son.columns]
if 'dmi_SON' in son:
    son['IOD'] = np.select([son.dmi_SON >= 0.4, son.dmi_SON <= -0.4], ['positif', 'negatif'], 'netral')
if 'oni_SON' in son:
    son['ENSO'] = np.select([son.oni_SON >= 0.5, son.oni_SON <= -0.5], ['El Niño', 'La Niña'], 'netral')
print('Rata-rata Sep-Nov per tahun (ambang kasar: DMI ±0,4 °C, ONI ±0,5):')
display(son)

fig, axes = plt.subplots(len(idx.columns), 1, figsize=(12, 2.6 * len(idx.columns)), sharex=True, constrained_layout=True)
for ax, c in zip(np.atleast_1d(axes), idx.columns):
    ax.fill_between(idx.index, 0, idx[c], where=idx[c] > 0, color='tab:red', alpha=.6, lw=0)
    ax.fill_between(idx.index, 0, idx[c], where=idx[c] < 0, color='tab:blue', alpha=.6, lw=0)
    ax.axhline(0, color='k', lw=.5); ax.set_ylabel(c.upper()); ax.grid(alpha=.25)
np.atleast_1d(axes)[0].set_title('Indeks iklim bulanan (merah = fase positif / El Niño)', fontsize=10)
plt.show()